## Cell 1: Setup
**What this demonstrates**: Temporal RAG requires no new dependencies — only the
standard Anthropic + OpenAI + ChromaDB stack with timestamp values stored as
integer metadata fields.
**Expected output**: Imports succeed; constants printed; decay function explained.

In [ ]:
# ── Install dependencies ─────────────────────────────────────────────────────
# Uncomment on first run:
# !pip install anthropic openai langchain-openai langchain-community chromadb \
#              python-dotenv --quiet

# ── Standard library ─────────────────────────────────────────────────────────
from __future__ import annotations
import math
import os
from dataclasses import dataclass, field
from datetime import date, datetime, timezone
from typing import Any

# ── Third-party ──────────────────────────────────────────────────────────────
from anthropic import Anthropic
from dotenv import load_dotenv
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

load_dotenv()

# ── API clients ───────────────────────────────────────────────────────────────
client     = Anthropic()   # reads ANTHROPIC_API_KEY
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')  # reads OPENAI_API_KEY

# ── Constants ─────────────────────────────────────────────────────────────────
SONNET  = 'claude-sonnet-4-6'
HAIKU   = 'claude-haiku-4-5-20251001'
TOP_K   = 3    # candidates retrieved before re-scoring

# Time-decay parameter lambda.
# decay_score = semantic_score * exp(-LAMBDA * age_in_days)
# LAMBDA = 0.001  → half-life ~693 days  (regulatory text: slow decay)
# LAMBDA = 0.005  → half-life ~139 days  (internal policy: moderate decay)
# LAMBDA = 0.05   → half-life ~14 days   (market data: fast decay)
LAMBDA_REGULATORY = 0.001
LAMBDA_POLICY     = 0.005

# Reference date: 'today' for decay calculations
REFERENCE_DATE = date(2024, 6, 1)  # fixed so demo output is reproducible


def to_epoch(d: date) -> int:
    '''Convert a date to a Unix epoch integer for ChromaDB where-clause comparisons.'''
    return int(datetime(d.year, d.month, d.day, tzinfo=timezone.utc).timestamp())


def age_in_days(doc_date: date, reference: date = REFERENCE_DATE) -> int:
    '''Number of days between doc_date and reference date.'''
    return max(0, (reference - doc_date).days)


def time_decay(semantic_score: float, doc_date: date,
               lam: float = LAMBDA_REGULATORY) -> float:
    '''Re-score a retrieved chunk by multiplying semantic similarity with a

    recency factor.  Exponential decay: score * exp(-lambda * age_in_days).
    Higher lambda = faster decay = stronger recency bias.
    '''
    days = age_in_days(doc_date)
    return semantic_score * math.exp(-lam * days)


print('Temporal RAG — Module 26')
print(f'  Model           : {SONNET}')
print(f'  Reference date  : {REFERENCE_DATE}')
print(f'  TOP_K           : {TOP_K}')
print()
print('Decay half-lives:')
print(f'  LAMBDA=0.001  → {math.log(2)/0.001:.0f} days  (regulatory text)')
print(f'  LAMBDA=0.005  → {math.log(2)/0.005:.0f} days  (internal policy)')
print(f'  LAMBDA=0.05   → {math.log(2)/0.05:.0f} days   (market data)')
print()
print('New in this module vs standard RAG:')
print('  + timestamp    : int metadata field (Unix epoch)')
print('  + superseded   : int metadata field (0=active, 1=superseded)')
print('  + effective_from: str metadata field (ISO date)')
print('  + time_decay() : post-retrieval re-scoring function')
print('  Everything else is unchanged from standard dense retrieval.')


## Cell 2: Data — Multi-Version Basel Regulatory Corpus (2013–2024)
**What this demonstrates**: A corpus where the same regulatory topic exists across
multiple versions with explicit supersession relationships. Standard RAG treats all
versions as equally valid. Temporal RAG uses the supersession chain to return only
the version current at the time of the query.
**Expected output**: Document timeline printed; supersession chains shown.

In [ ]:
# Multi-version Basel III/IV capital requirement corpus.
# Each document has: effective_date, superseded (0/1), superseded_by, doc_family.
# doc_family groups documents that represent the same regulatory topic over time.
#
# Supersession chains:
#   CET1 : doc_2013_cet1 → doc_2015_cet1 → doc_2019_cet1 → doc_2023_cet1 (active)
#   Leverage: doc_2014_leverage → doc_2019_leverage → doc_2023_leverage (active)
#   LCR  : doc_2019_lcr (active)
#   NSFR : doc_2021_nsfr (active)

DOCS: list[dict[str, Any]] = [
    # ── CET1 chain ────────────────────────────────────────────────────────────
    {
        'id': 'doc_2013_cet1',
        'effective_date': date(2013, 1, 1),
        'doc_family': 'cet1_minimum',
        'superseded': 1,
        'superseded_by': 'doc_2015_cet1',
        'text': (
            'Basel III CET1 Minimum Requirement — Effective 1 January 2013. '
            'Phase-in schedule: Common Equity Tier 1 (CET1) minimum ratio of 3.5 percent '
            'of risk-weighted assets (RWA). The capital conservation buffer of 0.625 percent '
            'begins phasing in from this date. Instruments not meeting CET1 criteria are '
            'subject to grandfathering provisions until January 2022. '
            'Banks must calculate CET1 using the standardised or internal ratings-based approach.'
        ),
    },
    {
        'id': 'doc_2015_cet1',
        'effective_date': date(2015, 1, 1),
        'doc_family': 'cet1_minimum',
        'superseded': 1,
        'superseded_by': 'doc_2019_cet1',
        'text': (
            'Basel III CET1 Update — Effective 1 January 2015. '
            'CET1 minimum ratio raised to 4.0 percent of RWA as part of the agreed phase-in. '
            'The capital conservation buffer increases to 1.25 percent. '
            'Deductions from CET1 (deferred tax assets, mortgage servicing rights, '
            'investments in financial institutions) are now applied at 40 percent of the '
            'fully phased-in amount. Dividend restrictions apply to banks below the buffer threshold.'
        ),
    },
    {
        'id': 'doc_2019_cet1',
        'effective_date': date(2019, 1, 1),
        'doc_family': 'cet1_minimum',
        'superseded': 1,
        'superseded_by': 'doc_2023_cet1',
        'text': (
            'Basel III CET1 Final Phase-In — Effective 1 January 2019. '
            'CET1 minimum ratio reaches its fully phased-in level of 4.5 percent of RWA. '
            'Capital conservation buffer fully phased in at 2.5 percent, bringing the '
            'effective CET1 target to 7.0 percent including the buffer. '
            'Countercyclical capital buffer of up to 2.5 percent may be imposed by national '
            'authorities. G-SIB surcharges of 1.0 to 3.5 percent apply to systemically '
            'important institutions.'
        ),
    },
    {
        'id': 'doc_2023_cet1',
        'effective_date': date(2023, 1, 1),
        'doc_family': 'cet1_minimum',
        'superseded': 0,
        'superseded_by': '',
        'text': (
            'Basel IV CET1 Framework — Effective 1 January 2023 (jurisdiction-dependent). '
            'CET1 minimum ratio maintained at 4.5 percent of RWA. '
            'Key change: revised standardised approach for credit risk replaces existing SA, '
            'affecting RWA calculation significantly. Output floor introduced at 72.5 percent: '
            'internal model RWA may not fall below 72.5 percent of SA RWA. '
            'Operational risk capital charge revised. CVA risk framework updated. '
            'Full implementation expected by 2028 for the output floor.'
        ),
    },
    # ── Leverage ratio chain ───────────────────────────────────────────────────
    {
        'id': 'doc_2014_leverage',
        'effective_date': date(2014, 1, 1),
        'doc_family': 'leverage_ratio',
        'superseded': 1,
        'superseded_by': 'doc_2019_leverage',
        'text': (
            'Basel III Leverage Ratio — Monitoring Period 2014. '
            'The leverage ratio is defined as Tier 1 capital divided by total exposure measure. '
            'Indicative minimum of 3 percent during the parallel run period. '
            'Public disclosure required from January 2015. '
            'Final calibration to be agreed by 2017 for migration to Pillar 1 in 2018.'
        ),
    },
    {
        'id': 'doc_2019_leverage',
        'effective_date': date(2019, 1, 1),
        'doc_family': 'leverage_ratio',
        'superseded': 1,
        'superseded_by': 'doc_2023_leverage',
        'text': (
            'Basel III Leverage Ratio — Pillar 1 Minimum, Effective 2019. '
            'Minimum leverage ratio of 3.0 percent of total exposures becomes a Pillar 1 '
            'requirement binding from 1 January 2019. '
            'Exposure measure includes on-balance sheet assets, derivatives exposure '
            '(replacement cost plus potential future exposure), securities financing '
            'transactions, and off-balance sheet items. '
            'G-SIBs are subject to additional leverage ratio buffer requirements.'
        ),
    },
    {
        'id': 'doc_2023_leverage',
        'effective_date': date(2023, 1, 1),
        'doc_family': 'leverage_ratio',
        'superseded': 0,
        'superseded_by': '',
        'text': (
            'Basel IV Leverage Ratio Update — Effective 1 January 2023. '
            'Minimum leverage ratio maintained at 3.0 percent for all banks. '
            'G-SIB leverage ratio buffer set at 50 percent of the G-SIB risk-based '
            'capital surcharge. A G-SIB with a 2.0 percent surcharge now faces a '
            '1.0 percent additional leverage ratio buffer. '
            'Revised exposure measure for derivatives under SA-CCR replaces CEM. '
            'Netting of client-cleared derivatives allowed for centrally cleared transactions.'
        ),
    },
    # ── LCR (single version, active) ──────────────────────────────────────────
    {
        'id': 'doc_2019_lcr',
        'effective_date': date(2019, 1, 1),
        'doc_family': 'lcr_requirement',
        'superseded': 0,
        'superseded_by': '',
        'text': (
            'Liquidity Coverage Ratio — Fully Phased In, Effective 1 January 2019. '
            'Banks must maintain a stock of high-quality liquid assets (HQLA) '
            'sufficient to cover net cash outflows over a 30-day stress scenario. '
            'Minimum LCR: 100 percent. HQLA is divided into Level 1 (0 percent haircut: '
            'central bank reserves, sovereign debt) and Level 2A/2B (15 to 50 percent '
            'haircut). Retail deposit run-off rates: 3 to 10 percent; wholesale: 25 to 100 percent.'
        ),
    },
    # ── NSFR (single version, active) ─────────────────────────────────────────
    {
        'id': 'doc_2021_nsfr',
        'effective_date': date(2021, 1, 1),
        'doc_family': 'nsfr_requirement',
        'superseded': 0,
        'superseded_by': '',
        'text': (
            'Net Stable Funding Ratio — Minimum Standard, Effective 1 January 2021. '
            'The NSFR requires banks to maintain a stable funding profile relative to '
            'the composition of their assets and off-balance sheet activities. '
            'Minimum NSFR: 100 percent (Available Stable Funding / Required Stable Funding). '
            'Long-term assets attract higher RSF factors: unencumbered residential '
            'mortgages carry a 65 percent RSF factor; other retail loans 85 percent; '
            'corporate loans 65 to 100 percent depending on maturity.'
        ),
    },
]

# Display corpus timeline
print('Basel III/IV regulatory corpus:')
print(f'  {len(DOCS)} documents across {len(set(d["doc_family"] for d in DOCS))} regulatory topics')
print()
print(f"  {'ID':<22} {'Effective':<14} {'Family':<18} {'Status'}")
print('  ' + '-' * 72)
for doc in sorted(DOCS, key=lambda d: d['effective_date']):
    status = 'SUPERSEDED' if doc['superseded'] else 'ACTIVE'
    arrow  = f" → {doc['superseded_by']}" if doc['superseded_by'] else ''
    print(f"  {doc['id']:<22} {str(doc['effective_date']):<14} "
          f"{doc['doc_family']:<18} {status}{arrow}")

print()
print('Supersession chains:')
print('  CET1     : doc_2013 → doc_2015 → doc_2019 → doc_2023 (ACTIVE)')
print('  Leverage : doc_2014 → doc_2019 → doc_2023 (ACTIVE)')
print('  LCR      : doc_2019 (ACTIVE, no amendments)')
print('  NSFR     : doc_2021 (ACTIVE, no amendments)')


## Cell 3: Core — Temporal Index and Retrieval Functions
**What this demonstrates**: Index documents with timestamp integer metadata for
ChromaDB where-clause filtering. Implement three retrieval modes: hard time filter,
time-decay re-scoring, and version-aware (suppress superseded). A query router
selects the appropriate mode from keyword signals in the query.
**Expected output**: Index built; all retrieval functions ready; query router shown.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Part A: Build the temporal index
# ─────────────────────────────────────────────────────────────────────────────

def build_temporal_index(docs: list[dict[str, Any]]) -> Chroma:
    '''Embed documents and store with temporal metadata in ChromaDB.

    Metadata fields per chunk:
      timestamp      : int  — Unix epoch of effective_date (for where-clause)
      effective_date : str  — ISO date string (for display)
      superseded     : int  — 0=active, 1=superseded
      superseded_by  : str  — doc_id of successor, or empty string
      doc_family     : str  — groups versions of the same regulation
    '''
    texts     = []
    ids       = []
    metadatas = []

    for doc in docs:
        # Prepend effective date to the text so the embedding reflects the version
        embed_text = (
            f"[Effective {doc['effective_date']}] "
            f"{'[SUPERSEDED] ' if doc['superseded'] else ''}"
            f"{doc['text']}"
        )
        texts.append(embed_text)
        ids.append(doc['id'])
        metadatas.append({
            'timestamp':      to_epoch(doc['effective_date']),
            'effective_date': str(doc['effective_date']),
            'superseded':     doc['superseded'],       # 0 or 1
            'superseded_by':  doc['superseded_by'],
            'doc_family':     doc['doc_family'],
        })

    return Chroma.from_texts(
        texts=texts,
        embedding=embeddings,
        ids=ids,
        metadatas=metadatas,
        collection_name='temporal_rag_demo',
    )


TEMPORAL_INDEX = build_temporal_index(DOCS)
print(f'Temporal index built: {len(DOCS)} documents')


# ─────────────────────────────────────────────────────────────────────────────
# Part B: Retrieve helpers — date lookup
# ─────────────────────────────────────────────────────────────────────────────

def doc_date_by_id(doc_id: str) -> date:
    '''Look up a document effective date from the DOCS list.'''
    for doc in DOCS:
        if doc['id'] == doc_id:
            return doc['effective_date']
    return REFERENCE_DATE  # fallback


# ─────────────────────────────────────────────────────────────────────────────
# Part C: Hard time filter retrieval
# ─────────────────────────────────────────────────────────────────────────────

def retrieve_before(query: str, cutoff: date, k: int = TOP_K) -> list[dict[str, Any]]:
    '''Retrieve documents with effective_date BEFORE the cutoff date.

    Uses a ChromaDB where-clause on the integer timestamp field.
    '''
    results = TEMPORAL_INDEX.similarity_search_with_score(
        query, k=k,
        filter={'timestamp': {'$lt': to_epoch(cutoff)}},
    )
    return [
        {'id': doc.metadata.get('doc_id', doc.id if hasattr(doc, 'id') else '?'),
         'text': doc.page_content,
         'score': float(s),
         'effective_date': doc.metadata.get('effective_date', '?'),
         'superseded': doc.metadata.get('superseded', -1),
         'doc_family': doc.metadata.get('doc_family', '?')}
        for doc, s in results
    ]


def retrieve_after(query: str, cutoff: date, k: int = TOP_K) -> list[dict[str, Any]]:
    '''Retrieve documents with effective_date ON OR AFTER the cutoff date.'''
    results = TEMPORAL_INDEX.similarity_search_with_score(
        query, k=k,
        filter={'timestamp': {'$gte': to_epoch(cutoff)}},
    )
    return [
        {'id': doc.metadata.get('doc_id', doc.id if hasattr(doc, 'id') else '?'),
         'text': doc.page_content,
         'score': float(s),
         'effective_date': doc.metadata.get('effective_date', '?'),
         'superseded': doc.metadata.get('superseded', -1),
         'doc_family': doc.metadata.get('doc_family', '?')}
        for doc, s in results
    ]


# ─────────────────────────────────────────────────────────────────────────────
# Part D: Time-decay re-scoring
# ─────────────────────────────────────────────────────────────────────────────

def retrieve_with_decay(
    query: str,
    k: int = TOP_K,
    lam: float = LAMBDA_REGULATORY,
) -> list[dict[str, Any]]:
    '''Retrieve top-k then re-score by multiplying semantic score with decay factor.

    Retrieves 2*k candidates first to give re-scoring enough material to work with.
    '''
    candidates = TEMPORAL_INDEX.similarity_search_with_score(query, k=k * 2)
    scored = []
    for doc, sem_score in candidates:
        eff_date_str = doc.metadata.get('effective_date', str(REFERENCE_DATE))
        eff_date = date.fromisoformat(eff_date_str)
        decay_factor = math.exp(-lam * age_in_days(eff_date))
        final_score  = float(sem_score) * decay_factor
        scored.append({
            'id': doc.metadata.get('doc_id', doc.id if hasattr(doc, 'id') else '?'),
            'text': doc.page_content,
            'semantic_score': float(sem_score),
            'decay_factor':   round(decay_factor, 4),
            'final_score':    round(final_score, 4),
            'effective_date': eff_date_str,
            'superseded':     doc.metadata.get('superseded', -1),
            'doc_family':     doc.metadata.get('doc_family', '?'),
        })
    # Sort by final_score ascending (ChromaDB returns distance: lower = better)
    scored.sort(key=lambda x: x['final_score'])
    return scored[:k]


# ─────────────────────────────────────────────────────────────────────────────
# Part E: Version-aware retrieval (active versions only)
# ─────────────────────────────────────────────────────────────────────────────

def retrieve_active(query: str, k: int = TOP_K) -> list[dict[str, Any]]:
    '''Retrieve only non-superseded (active) documents.

    Uses a where-clause filter on the superseded integer field.
    Returns the current version of each regulatory topic.
    '''
    results = TEMPORAL_INDEX.similarity_search_with_score(
        query, k=k,
        filter={'superseded': {'$eq': 0}},
    )
    return [
        {'id': doc.metadata.get('doc_id', doc.id if hasattr(doc, 'id') else '?'),
         'text': doc.page_content,
         'score': float(s),
         'effective_date': doc.metadata.get('effective_date', '?'),
         'doc_family': doc.metadata.get('doc_family', '?')}
        for doc, s in results
    ]


# ─────────────────────────────────────────────────────────────────────────────
# Part F: Query router — detect temporal intent from query text
# ─────────────────────────────────────────────────────────────────────────────

import re as _re

def detect_temporal_intent(query: str) -> str:
    '''Classify query into one of four temporal retrieval modes.

    Returns: 'current', 'historical', 'comparative', or 'range'.
    '''
    q = query.lower()
    comparative_signals = ['before and after', 'before vs after', 'change between',
                           'how did', 'evolution', 'history of', 'compare']
    historical_signals  = ['before', 'prior to', 'as of', 'in 201', 'in 202',
                           'was the', 'used to', 'originally', 'historically']
    current_signals     = ['current', 'latest', 'now', 'today', 'active',
                           'in force', 'effective today']

    if any(s in q for s in comparative_signals):
        return 'comparative'
    if any(s in q for s in historical_signals):
        return 'historical'
    if any(s in q for s in current_signals):
        return 'current'
    return 'current'  # safe default: return active versions


print()
print('All temporal retrieval functions ready:')
print('  retrieve_before(query, cutoff_date)  — hard filter: before date')
print('  retrieve_after(query, cutoff_date)   — hard filter: on/after date')
print('  retrieve_with_decay(query, lambda)   — recency-weighted re-scoring')
print('  retrieve_active(query)               — version-aware: active only')
print('  detect_temporal_intent(query)        — route to correct mode')

# Quick intent classification demo
test_queries = [
    'What is the current CET1 minimum?',
    'What was the leverage ratio requirement before 2019?',
    'How did the CET1 requirement change between 2015 and 2023?',
]
print()
print('Intent classification examples:')
for tq in test_queries:
    intent = detect_temporal_intent(tq)
    print(f'  [{intent:>12}]  {tq}')


## Cell 4: Run — Before/After Comparative Query
**What this demonstrates**: A comparative query that requires two separate
time-filtered retrievals — one before 2023 and one after — merged into a
single context for before/after synthesis. Standard RAG cannot answer this
query reliably; it would return whichever version scored highest.
**Expected output**: Before/after documents retrieved; comparative answer with dates.

In [ ]:
# Comparative query: requires retrieving the pre-2023 and post-2023 versions
# separately and presenting both to the LLM for a structured comparison.

QUERY = (
    'What was the capital adequacy requirement before vs after the 2023 Basel update? '
    'How did the CET1 ratio and leverage ratio change?'
)

CHANGE_DATE = date(2023, 1, 1)

print(f'Query : {QUERY}')
print(f'Change date : {CHANGE_DATE}')
print('=' * 70)

# ── Two-pass temporal retrieval ───────────────────────────────────────────────
before_results = retrieve_before(QUERY, cutoff=CHANGE_DATE, k=3)
after_results  = retrieve_after(QUERY,  cutoff=CHANGE_DATE, k=3)

print(f'\nBefore {CHANGE_DATE}: {len(before_results)} documents retrieved')
for r in before_results:
    status = 'superseded' if r['superseded'] else 'active'
    print(f"  [{r['effective_date']}] {r['doc_family']:<20} score={r['score']:.3f}  ({status})")

print(f'\nAfter {CHANGE_DATE}: {len(after_results)} documents retrieved')
for r in after_results:
    status = 'superseded' if r['superseded'] else 'active'
    print(f"  [{r['effective_date']}] {r['doc_family']:<20} score={r['score']:.3f}  ({status})")

# ── Assemble comparative context ──────────────────────────────────────────────
before_block = '\n\n'.join(
    f"[{r['effective_date']}]\n{r['text']}" for r in before_results
)
after_block = '\n\n'.join(
    f"[{r['effective_date']}]\n{r['text']}" for r in after_results
)

comparative_context = (
    '## PRE-2023 REQUIREMENTS\n\n' + before_block
    + '\n\n## POST-2023 REQUIREMENTS\n\n' + after_block
)

# ── Synthesise ────────────────────────────────────────────────────────────────
SYSTEM = (
    'You are a banking regulation analyst. '
    'Compare the requirements using ONLY the provided context. '
    'Structure your answer as:\n'
    '1. CET1 ratio: before → after\n'
    '2. Leverage ratio: before → after\n'
    '3. Key structural changes (not just ratio changes)\n'
    'Cite the effective date for every figure you reference.'
)

response = client.messages.create(
    model=SONNET,
    max_tokens=700,
    system=SYSTEM,
    messages=[{
        'role': 'user',
        'content': f'Question: {QUERY}\n\nContext:\n{comparative_context}',
    }],
)
answer = response.content[0].text

print('\n' + '=' * 70)
print('TEMPORAL RAG ANSWER')
print('=' * 70)
print(answer)


## Cell 5: Inspect — Decay Scores, Version Selection, and Flat Baseline
**What this demonstrates**: (1) Time-decay score table: same query, same semantic
similarity, different final scores based on document age. (2) Version-aware
retrieval suppressing superseded documents. (3) Standard (flat) retrieval baseline
showing stale documents surfacing without temporal filtering.
**Expected output**: Decay table; version chain; flat vs temporal comparison.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Part A: Time-decay score table
# ─────────────────────────────────────────────────────────────────────────────
INSPECT_QUERY = 'What is the minimum CET1 capital ratio requirement?'

print('== Time-decay score table ==')
print(f'  Query: {INSPECT_QUERY}')
print(f'  Reference date: {REFERENCE_DATE}   Lambda: {LAMBDA_REGULATORY}')
print()

decay_results = retrieve_with_decay(INSPECT_QUERY, k=len(DOCS), lam=LAMBDA_REGULATORY)

print(f"  {'Doc ID':<22} {'Eff. Date':<14} {'Sem. Score':>10} "
      f"{'Age (days)':>10} {'Decay':>7} {'Final Score':>12}")
print('  ' + '-' * 80)
for r in decay_results:
    eff = r['effective_date']
    doc_date = date.fromisoformat(eff)
    days = age_in_days(doc_date)
    marker = ' ← lowest score (oldest)' if r == decay_results[-1] else ''
    print(f"  {r['id']:<22} {eff:<14} {r['semantic_score']:>10.4f} "
          f"{days:>10} {r['decay_factor']:>7.4f} {r['final_score']:>12.4f}{marker}")

print()
print('Observation: the 2013 document has the same regulatory content as the')
print('2023 document — both mention CET1 requirements. The semantic scores are')
print('similar. The decay factor is what differentiates them: the 2013 document')
print(f'is ~{age_in_days(date(2013,1,1))} days old; its final score is penalised accordingly.')

# ─────────────────────────────────────────────────────────────────────────────
# Part B: Version-aware retrieval — active only
# ─────────────────────────────────────────────────────────────────────────────
print()
print('== Version-aware retrieval (superseded=0 filter) ==')
active_results = retrieve_active(INSPECT_QUERY, k=4)
print(f'  Retrieved {len(active_results)} active documents (superseded ones excluded):')
for r in active_results:
    print(f"  [{r['effective_date']}] {r['id']:<22} {r['doc_family']}")

# ─────────────────────────────────────────────────────────────────────────────
# Part C: Flat baseline — no temporal filtering (standard RAG behaviour)
# ─────────────────────────────────────────────────────────────────────────────
print()
print('== Standard RAG baseline (no temporal filter) ==')
flat_results = TEMPORAL_INDEX.similarity_search_with_score(INSPECT_QUERY, k=4)
print(f'  Retrieved {len(flat_results)} documents:')
for doc, score in flat_results:
    eff   = doc.metadata.get('effective_date', '?')
    sup   = doc.metadata.get('superseded', -1)
    fam   = doc.metadata.get('doc_family', '?')
    label = ' ← STALE' if sup == 1 else ' ← current'
    print(f'  [{eff}] score={score:.3f}  {fam}{label}')

# ─────────────────────────────────────────────────────────────────────────────
# Part D: Version chain validation
# ─────────────────────────────────────────────────────────────────────────────
print()
print('== Supersession chain: CET1 family ==')
cet1_docs = sorted(
    [d for d in DOCS if d['doc_family'] == 'cet1_minimum'],
    key=lambda d: d['effective_date']
)
for i, doc in enumerate(cet1_docs):
    arrow = f" → {doc['superseded_by']}" if doc['superseded_by'] else ' (ACTIVE)'
    print(f'  [{doc["effective_date"]}] {doc["id"]}{arrow}')

active_count = sum(1 for d in cet1_docs if not d['superseded'])
print(f'\n  Chain length : {len(cet1_docs)}')
print(f'  Active docs  : {active_count}  (should always be 1 per family)')


## Cell 6: Fintech — Regulatory Change Timeline Q&A
**What this demonstrates**: Three fintech query types that each exercise a
different temporal retrieval mode: (A) point-in-time audit query (hard filter),
(B) current requirement query (version-aware), (C) multi-period evolution
query (comparative two-pass). Each demonstrates a distinct use case for
temporal RAG in compliance and regulatory workflows.
**Expected output**: Three structured answers; temporal mode used shown for each.

In [ ]:
FINTECH_SYSTEM = (
    'You are a regulatory compliance analyst. '
    'Answer using ONLY the provided context. '
    'Cite the effective date of every requirement you state. '
    'If the context contains superseded documents, note that they have been replaced.'
)

def temporal_qa(
    query: str,
    mode: str,
    cutoff_before: date | None = None,
    cutoff_after:  date | None = None,
    max_tokens: int = 500,
) -> dict[str, Any]:
    '''Run a temporal query and return the answer with provenance metadata.'''
    if mode == 'current':
        results = retrieve_active(query, k=3)
        context = '\n\n'.join(
            f"[Active as of {r['effective_date']}]\n{r['text']}" for r in results
        )
    elif mode == 'historical' and cutoff_before:
        results = retrieve_before(query, cutoff=cutoff_before, k=3)
        context = '\n\n'.join(
            f"[Effective {r['effective_date']}]\n{r['text']}" for r in results
        )
    elif mode == 'comparative' and cutoff_after:
        before_r = retrieve_before(query, cutoff=cutoff_after, k=2)
        after_r  = retrieve_after(query,  cutoff=cutoff_after, k=2)
        results  = before_r + after_r
        context  = (
            '## Before\n\n'
            + '\n\n'.join(f"[{r['effective_date']}]\n{r['text']}" for r in before_r)
            + '\n\n## After\n\n'
            + '\n\n'.join(f"[{r['effective_date']}]\n{r['text']}" for r in after_r)
        )
    else:
        results = retrieve_active(query, k=3)
        context = '\n\n'.join(r['text'] for r in results)

    response = client.messages.create(
        model=SONNET,
        max_tokens=max_tokens,
        system=FINTECH_SYSTEM,
        messages=[{'role': 'user',
                   'content': f'Question: {query}\n\nContext:\n{context}'}],
    )
    return {
        'query': query,
        'mode': mode,
        'docs_retrieved': len(results),
        'answer': response.content[0].text,
    }


# ── Query A: Point-in-time audit (hard filter) ────────────────────────────────
print('=' * 70)
print('QUERY A — Point-in-time audit (historical hard filter)')
print('=' * 70)
QA = (
    'What CET1 and leverage ratio requirements were in effect during Q1 2016 '
    'for a bank preparing its annual regulatory report?'
)
result_a = temporal_qa(QA, mode='historical', cutoff_before=date(2016, 4, 1))
print(f'Mode: {result_a["mode"]}  |  Docs retrieved: {result_a["docs_retrieved"]}')
print(f'Query: {result_a["query"]}')
print()
print(result_a['answer'])

# ── Query B: Current requirement (version-aware) ──────────────────────────────
print()
print('=' * 70)
print('QUERY B — Current requirement (version-aware active filter)')
print('=' * 70)
QB = (
    'What are the current minimum capital ratios a bank must maintain '
    'under Basel regulations, including the G-SIB surcharge rules?'
)
result_b = temporal_qa(QB, mode='current')
print(f'Mode: {result_b["mode"]}  |  Docs retrieved: {result_b["docs_retrieved"]}')
print(f'Query: {result_b["query"]}')
print()
print(result_b['answer'])

# ── Query C: Multi-period evolution (comparative two-pass) ────────────────────
print()
print('=' * 70)
print('QUERY C — Regulatory evolution (comparative two-pass)')
print('=' * 70)
QC = (
    'How did Basel capital requirements evolve from before 2019 to after 2023? '
    'Focus on the leverage ratio and liquidity requirements.'
)
result_c = temporal_qa(QC, mode='comparative', cutoff_after=date(2019, 1, 1))
print(f'Mode: {result_c["mode"]}  |  Docs retrieved: {result_c["docs_retrieved"]}')
print(f'Query: {result_c["query"]}')
print()
print(result_c['answer'])

# ── Summary ───────────────────────────────────────────────────────────────────
print()
print('=' * 70)
print('TEMPORAL RETRIEVAL MODE SUMMARY')
print('=' * 70)
print()
print(f"  {'Query':<50} {'Mode':<14} {'Docs'}")
print('  ' + '-' * 70)
for res in [result_a, result_b, result_c]:
    label = res['query'][:48] + '..'
    print(f"  {label:<50} {res['mode']:<14} {res['docs_retrieved']}")

print()
print('Without temporal RAG, all three queries return the same pool of documents')
print('ranked by semantic similarity alone. The 2013 and 2015 circulars appear')
print('in all results regardless of whether they are current or superseded.')
print('Query A in particular would fail silently: standard RAG might return')
print('the 2023 Basel IV rule for a 2016 audit query, producing a wrong answer.')
